# arXiv

In [1]:
%reload_ext sql
%sql duckdb:///:memory:
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = False

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'duckdb:///:memory:'

In [2]:
%%sql

CREATE VIEW arXiv_pdfs AS
SELECT *
FROM read_parquet('../data/output/arXiv_pdf_manifest.parquet');

CREATE VIEW arXiv_src AS
SELECT *
FROM read_parquet('../data/output/arXiv_src_manifest.parquet');

CREATE VIEW arxiv_extract AS
SELECT *
FROM read_parquet('../data/output/results-2026-04-24/**/*.parquet');

CREATE VIEW arxiv_metadata AS
SELECT
    REPLACE(id, '/', '') AS arxiv_id,
    update_date,
    versions
FROM read_json_auto('../data/sources/arxiv-metadata-oai-snapshot.json')
WHERE try_strptime(
    (list_filter(versions, v -> v.version = 'v1'))[1].created,
    '%a, %-d %b %Y %H:%M:%S GMT'
) <= TIMESTAMP '2026-03-31 23:59:59';

-- 2026-01-01 00:00:00: only_in_arxiv_extract=76931, only_in_arxiv_metadata=0
-- 2026-03-31 00:00:00: only_in_arxiv_extract=937, only_in_arxiv_metadata=264
-- 2026-03-31 23:59:59: only_in_arxiv_extract=4, only_in_arxiv_metadata=563


Running query in 'duckdb:///:memory:'

Count


In [3]:
%%sql

SELECT
  'src' AS Directory,
  format('{:,}', SUM(num_items)) AS num_files,
  format('{:,}', COUNT(*)) AS num_tars,
  format('{:,.2f}', SUM(size) / POWER(1024, 4)) AS tars_tib,
  format('${:,.2f}', SUM(size) / POWER(1024, 3) * 0.09) AS internet_egress_usd
FROM arXiv_src

UNION ALL

SELECT
  'pdf' AS Directory,
  format('{:,}', SUM(num_items)) AS num_files,
  format('{:,}', COUNT(*)) AS num_tars,
  format('{:,.2f}', SUM(size) / POWER(1024, 4)) AS tars_tib,
  format('${:,.2f}', SUM(size) / POWER(1024, 3) * 0.09) AS internet_egress_usd
FROM arXiv_pdfs;

Running query in 'duckdb:///:memory:'

Directory,num_files,num_tars,tars_tib,internet_egress_usd
src,"2,999,810","12,173",5.90,$543.36
pdf,"2,975,873","11,286",5.44,$501.79


In [4]:
%%sql

SELECT                                                                                                                                                                                                                                
    CASE
        WHEN CAST(substr(yymm, 1, 2) AS INTEGER) <= 30
            THEN 2000 + CAST(substr(yymm, 1, 2) AS INTEGER)
        ELSE 1900 + CAST(substr(yymm, 1, 2) AS INTEGER)
    END AS year,                
    COUNT(*) AS num_tars,                                                                                                                                                                                                              
    ROUND(SUM(size) / 1e9, 2) AS total_gb,                                                                                                                                                                                              
    ROUND(AVG(size) / 1e6, 1) AS avg_size_mb,                                                                                                                                                                                           
    ROUND(MIN(size) / 1e6, 1) AS min_size_mb,                                                                                                                                                                                           
    ROUND(MAX(size) / 1e6, 1) AS max_size_mb
FROM arXiv_src                                                                                                                                                                                                                         
GROUP BY year                                             
ORDER BY year;

Running query in 'duckdb:///:memory:'

year,num_tars,total_gb,avg_size_mb,min_size_mb,max_size_mb
1991,6,0.01,1.0,0.1,1.9
1992,12,0.07,5.6,2.0,8.8
1993,12,0.21,17.2,9.5,28.3
1994,12,0.42,34.8,22.2,47.7
1995,12,0.67,56.2,45.8,63.5
1996,12,0.92,76.7,59.6,99.2
1997,12,1.37,114.1,76.8,141.6
1998,12,1.99,166.1,128.5,204.5
1999,12,2.46,205.2,162.6,244.7
2000,12,2.96,246.4,191.6,287.0


In [5]:
%%sql

SELECT                                                                                                                                                                                                                                
    strftime('%Y', timestamp) AS update_year,                      
    COUNT(*) AS num_tars,                                                                                                                                                                                                              
    ROUND(SUM(size) / 1e9, 2) AS total_gb,                                                                                                                                                                                              
    ROUND(AVG(size) / 1e6, 1) AS avg_size_mb,                                                                                                                                                                                           
    ROUND(MIN(size) / 1e6, 1) AS min_size_mb,                                                                                                                                                                                           
    ROUND(MAX(size) / 1e6, 1) AS max_size_mb
FROM arXiv_src                                                                                                                                                                                                                         
GROUP BY update_year                                             
ORDER BY update_year;

Running query in 'duckdb:///:memory:'

update_year,num_tars,total_gb,avg_size_mb,min_size_mb,max_size_mb
2010,318,96.8,304.4,1.0,576.1
2011,25,11.25,450.0,1.8,658.7
2012,92,43.14,468.9,0.1,636.3
2013,107,54.58,510.1,7.8,586.9
2014,148,76.76,518.6,71.4,687.2
2015,128,65.67,513.0,10.5,675.3
2016,184,98.35,534.5,79.2,1910.6
2017,228,120.02,526.4,33.0,661.7
2018,293,154.09,525.9,28.9,674.9
2019,334,178.99,535.9,38.7,851.6


In [6]:
%%sql

select status, file_type, COUNT(*) AS count 
FROM arxiv_extract
GROUP BY status, file_type
ORDER BY COUNT DESC;

Running query in 'duckdb:///:memory:'

status,file_type,count
ok,tex,2736025
empty,pdf,244891
empty,tex,12531
empty,postscript,4356
empty,html,943
timeout,tex,499
archive_error,unknown,340
skipped,tex,199
archive_error,pdf,25
empty,unknown,1


In [7]:
%%sql

select status, COUNT(*) AS count 
FROM arxiv_extract
GROUP BY status
ORDER BY COUNT DESC;

Running query in 'duckdb:///:memory:'

status,count
ok,2736025
empty,262722
timeout,499
archive_error,365
skipped,199


In [8]:
%%sql

SELECT
  CASE
    WHEN status = 'ok' THEN 'ok'
    ELSE 'not_ok'
  END AS status_group,
  COUNT(*) AS count
FROM arxiv_extract
GROUP BY 1
ORDER BY 1;

Running query in 'duckdb:///:memory:'

status_group,count
not_ok,263785
ok,2736025


In [9]:
%%sql
    
SELECT COUNT(DISTINCT arxiv_id) FROM arxiv_metadata;

Running query in 'duckdb:///:memory:'

count(DISTINCT arxiv_id)
3000369


In [10]:
%%sql

WITH a AS (
  SELECT DISTINCT arxiv_id
  FROM arxiv_extract
  WHERE arxiv_id IS NOT NULL
),
b AS (
  SELECT DISTINCT arxiv_id
  FROM arxiv_metadata
  WHERE arxiv_id IS NOT NULL
)
SELECT
  CASE
    WHEN a.arxiv_id IS NOT NULL AND b.arxiv_id IS NOT NULL THEN 'in_both'
    WHEN a.arxiv_id IS NOT NULL THEN 'only_in_arxiv_extract'
    ELSE 'only_in_arxiv_metadata'
  END AS membership,
  COUNT(*) AS count
FROM a
FULL OUTER JOIN b
  ON a.arxiv_id = b.arxiv_id
GROUP BY 1
ORDER BY 1;

Running query in 'duckdb:///:memory:'

membership,count
in_both,2999806
only_in_arxiv_extract,4
only_in_arxiv_metadata,563


In [11]:
%%sql

WITH a AS (
  SELECT DISTINCT arxiv_id
  FROM arxiv_extract
  WHERE arxiv_id IS NOT NULL
),
b AS (
  SELECT DISTINCT arxiv_id
  FROM arxiv_metadata
  WHERE arxiv_id IS NOT NULL
)
SELECT a.arxiv_id
FROM a
LEFT JOIN b
  ON a.arxiv_id = b.arxiv_id
WHERE b.arxiv_id IS NULL
ORDER BY a.arxiv_id
LIMIT 100;

Running query in 'duckdb:///:memory:'

arxiv_id
2307.02646
2401.09755
2402.18611
acc-phys9607002


In [12]:
%%sql

WITH extract_status AS (
  SELECT
    arxiv_id,
    CASE
      WHEN status = 'ok' THEN 'ok'
      ELSE 'not_ok'
    END AS status_group
  FROM arxiv_extract
  WHERE arxiv_id IS NOT NULL
),
metadata_classified AS (
  SELECT
    m.arxiv_id,
    CASE
      WHEN e.arxiv_id IS NULL THEN 'not_in_arxiv_extract'
      WHEN e.status_group = 'not_ok' THEN 'not_ok_in_arxiv_extract'
      ELSE 'ok_in_arxiv_extract'
    END AS membership
  FROM arxiv_metadata m
  LEFT JOIN extract_status e
    ON m.arxiv_id = e.arxiv_id
  WHERE m.arxiv_id IS NOT NULL
)
SELECT
  membership,
  COUNT(*) AS count
FROM metadata_classified
WHERE membership IN ('not_ok_in_arxiv_extract', 'not_in_arxiv_extract')
GROUP BY 1

UNION ALL

SELECT
  'total_sum' AS membership,
  COUNT(*) AS count
FROM metadata_classified
WHERE membership IN ('not_ok_in_arxiv_extract', 'not_in_arxiv_extract');

Running query in 'duckdb:///:memory:'

membership,count
not_in_arxiv_extract,563
not_ok_in_arxiv_extract,263786
total_sum,264349
